In [ ]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
import glob
import shutil
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Conv2D, BatchNormalization, Dropout, Activation, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [ ]:

# ==========================================
# 1. SETUP & DATA GENERATOR
# ==========================================
OUTPUT_DIR = '/kaggle/working/hasil_skenario'
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

DATASET_PATH = '/kaggle/input/fake-vs-real-dataset/real-vs-fake10k' 
BATCH_SIZE = 32
IMG_SIZE = (224, 224)

train_datagen = ImageDataGenerator(rescale=1./255, rotation_range=20, horizontal_flip=True)
valid_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    os.path.join(DATASET_PATH, 'train'),
    target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary'
)
valid_gen = valid_datagen.flow_from_directory(
    os.path.join(DATASET_PATH, 'valid'),
    target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary'
)

# TEST GENERATOR
test_gen = valid_datagen.flow_from_directory(
    os.path.join(DATASET_PATH, 'test'),
    target_size=IMG_SIZE, batch_size=1, class_mode='binary', shuffle=False
)



In [ ]:

# ==========================================
# 2. FUNGSI VISUALISASI 
# ==========================================

# A. Fungsi Plot Grafik Training (Loss & Accuracy)
def plot_training_graphs(history, model_name):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_range = range(len(acc))

    plt.figure(figsize=(12, 5))
    
    # Plot 1: Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy')
    plt.title(f'{model_name} - Accuracy')
    plt.legend(loc='lower right')
    plt.grid(True)

    # Plot 2: Loss
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss')
    plt.plot(epochs_range, val_loss, label='Validation Loss')
    plt.title(f'{model_name} - Loss')
    plt.legend(loc='upper right')
    plt.grid(True)
    
    plt.tight_layout()
    # Simpan Gambar Grafik
    plt.savefig(os.path.join(OUTPUT_DIR, f"{model_name}_graph.png"))
    plt.close()
    print(f"Grafik Training tersimpan: {model_name}_graph.png")

# B. Fungsi Evaluasi (Confusion Matrix & Report)
def evaluate_and_save(model, model_name):
    print(f"Mengevaluasi {model_name} pada Data Test...")
    
    # Prediksi
    test_gen.reset()
    predictions = model.predict(test_gen, verbose=1)
    y_pred = (predictions > 0.5).astype(int).flatten()
    y_true = test_gen.classes
    
    # 1. Simpan Classification Report (CSV)
    report_dict = classification_report(y_true, y_pred, target_names=['Fake', 'Real'], output_dict=True)
    df_report = pd.DataFrame(report_dict).transpose()
    df_report.to_csv(os.path.join(OUTPUT_DIR, f"{model_name}_report.csv"))
    
    # 2. Simpan Confusion Matrix (PNG)
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Fake', 'Real'], yticklabels=['Fake', 'Real'])
    plt.title(f'CM: {model_name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{model_name}_CM.png")) 
    plt.close()
    
    print(f"Evaluasi selesai untuk {model_name}")

In [ ]:
# ==========================================
# 3. FUNGSI BUILD MODEL
# ==========================================
def build_model(use_conv, kernel_size, filters, dense_units, dropout, fine_tune):
    input_tensor = Input(shape=(224, 224, 3))
    base_model = MobileNetV2(weights="imagenet", include_top=False, input_tensor=input_tensor)
    
    if fine_tune:
        base_model.trainable = True
        fine_tune_at = len(base_model.layers) // 2
        for layer in base_model.layers[:fine_tune_at]:
            layer.trainable = False
    else:
        base_model.trainable = False

    x = base_model.output

    if use_conv:
        x = Conv2D(filters, kernel_size, padding='same', name='custom_conv')(x)
        x = BatchNormalization()(x)
        x = Activation('relu')(x)
    
    x = GlobalAveragePooling2D()(x)
    x = Dense(dense_units, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(dropout)(x)
    output = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=base_model.input, outputs=output)
    lr = 1e-5 if fine_tune else 1e-4
    model.compile(loss="binary_crossentropy", optimizer=Adam(learning_rate=lr), metrics=["accuracy"])
    return model


In [ ]:
#LOOPING UNTUK 12 SKENARIO BERDASAERKAN PENDEKATAN YANG SUDAH DITENTUKAN
# ==========================================
# 4. JALANKAN SKENARIO 1 (Head + Dense)
# ==========================================
print("\nMEMULAI SKENARIO 1 (6 Kombinasi)...")
dense_options = [256, 512]
dropout_options = [0.3, 0.5, 0.7]

for d_unit in dense_options:
    for drop in dropout_options:
        model_name = f"S1_Dense{d_unit}_Drop{drop}"
        print(f"\nTraining: {model_name} ...")
        
        model = build_model(use_conv=False, kernel_size=None, filters=0, dense_units=d_unit, dropout=drop, fine_tune=False)
        es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        history = model.fit(train_gen, epochs=50, validation_data=valid_gen, callbacks=[es], verbose=1)
        
        # Simpan Model & History Data
        model.save(os.path.join(OUTPUT_DIR, f"{model_name}.keras"))
        pd.DataFrame(history.history).to_csv(os.path.join(OUTPUT_DIR, f"{model_name}_history.csv"))
        
        # --- VISUALISASI (GAMBAR TERPISAH) ---
        plot_training_graphs(history, model_name) # Grafik Garis
        evaluate_and_save(model, model_name)      # Confusion Matrix & Report

# ==========================================
# 5. JALANKAN SKENARIO 2 (Head + Conv)
# ==========================================
print("\nMEMULAI SKENARIO 2 (4 Kombinasi)...")
kernel_options = [(3,3), (7,7)]
filter_options = [256, 512]

for k_size in kernel_options:
    for filt in filter_options:
        model_name = f"S2_Conv_K{k_size[0]}x{k_size[1]}_F{filt}"
        print(f"\nTraining: {model_name} ...")
        
        model = build_model(use_conv=True, kernel_size=k_size, filters=filt, dense_units=512, dropout=0.5, fine_tune=False)
        es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        history = model.fit(train_gen, epochs=50, validation_data=valid_gen, callbacks=[es], verbose=1)
        
        model.save(os.path.join(OUTPUT_DIR, f"{model_name}.keras"))
        pd.DataFrame(history.history).to_csv(os.path.join(OUTPUT_DIR, f"{model_name}_history.csv"))
        
        # --- VISUALISASI (GAMBAR TERPISAH) ---
        plot_training_graphs(history, model_name)
        evaluate_and_save(model, model_name)

# ==========================================
# 6. ANALISA PEMENANG
# ==========================================
print("\nMenganalisa Pemenang...")
hist_files = glob.glob(os.path.join(OUTPUT_DIR, "*_history.csv"))
best_s1_acc = 0; best_s1_name = ""
best_s2_acc = 0; best_s2_name = ""

for f in hist_files:
    df = pd.read_csv(f); max_acc = df['val_accuracy'].max()
    fname = os.path.basename(f).replace("_history.csv", "")
    if "S1_" in fname and max_acc > best_s1_acc: best_s1_acc = max_acc; best_s1_name = fname
    if "S2_" in fname and max_acc > best_s2_acc: best_s2_acc = max_acc; best_s2_name = fname

print(f"Best Model S1: {best_s1_name} ({best_s1_acc:.4f})")
print(f"Bewt Model S2: {best_s2_name} ({best_s2_acc:.4f})")

# ==========================================
# 7. SKENARIO 3 & 4 (FINE TUNING)
# ==========================================
def parse_config(name):
    config = {'fine_tune': True}
    if "S1_" in name:
        parts = name.split('_')
        config.update({'use_conv': False, 'kernel_size': None, 'filters': 0, 
                       'dense_units': int(parts[1].replace('Dense','')), 'dropout': float(parts[2].replace('Drop',''))})
    elif "S2_" in name:
        parts = name.split('_'); k_str = parts[2].replace('K','').split('x')
        config.update({'use_conv': True, 'kernel_size': (int(k_str[0]), int(k_str[1])), 
                       'filters': int(parts[3].replace('F','')), 'dense_units': 512, 'dropout': 0.5})
    return config

# Skenario 3
name_s3 = f"S3_FineTune_{best_s1_name}"
print(f"\n Training S3: {name_s3} ...")
model_s3 = build_model(**parse_config(best_s1_name))
es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
hist_s3 = model_s3.fit(train_gen, epochs=50, validation_data=valid_gen, callbacks=[es], verbose=1)

model_s3.save(os.path.join(OUTPUT_DIR, f"{name_s3}.keras"))
pd.DataFrame(hist_s3.history).to_csv(os.path.join(OUTPUT_DIR, f"{name_s3}_history.csv"))
# Visualisasi S3
plot_training_graphs(hist_s3, name_s3)
evaluate_and_save(model_s3, name_s3)

# Skenario 4
name_s4 = f"S4_FineTune_{best_s2_name}"
print(f"\n Training S4: {name_s4} ...")
model_s4 = build_model(**parse_config(best_s2_name))
es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
hist_s4 = model_s4.fit(train_gen, epochs=50, validation_data=valid_gen, callbacks=[es], verbose=1)

model_s4.save(os.path.join(OUTPUT_DIR, f"{name_s4}.keras"))
pd.DataFrame(hist_s4.history).to_csv(os.path.join(OUTPUT_DIR, f"{name_s4}_history.csv"))
# Visualisasi S4
plot_training_graphs(hist_s4, name_s4)
evaluate_and_save(model_s4, name_s4)



In [ ]:
# ==========================================
# 8. ZIP & SELESAI
# ==========================================

shutil.make_archive("Hasil_Lengkap_Skenario", 'zip', OUTPUT_DIR)
print("'Hasil_Lengkap_Skenario.zip' siap di Output.")

In [1]:
#VISUALISASI 

import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. KONFIGURASI LOKASI PENCARIAN (UPDATED)
# ==========================================

SEARCH_DIRS = [ 
    '/kaggle/input/hasil_mobilenetv2', #Donwload ZIP hasil luaran training dan uploa sebagai dataset baru, add input ke notebook baru bisa di run kode ini, astikan path nya sama
]

print("🔍 Memulai Operasi Rekapitulasi Data...")
print(f"📂 Mencari di folder: {SEARCH_DIRS}")

# ==========================================
# 2. LOGIKA PENGUMPULAN DATA
# ==========================================
recap_data = []

# Cari semua file report (.csv)
# Menggunakan recursive=True agar bisa menembus sub-folder di dalam ZIP
report_files = []
for d in SEARCH_DIRS:
    # Cari file berakhiran _report.csv di dalam folder dan sub-foldernya
    found = glob.glob(os.path.join(d, "**/*_report.csv"), recursive=True)
    report_files.extend(found)

# Hilangkan duplikat dan urutkan
report_files = list(set(report_files))
report_files.sort()

if not report_files:
    print("\n ERROR: Tidak ada file '_report.csv' ditemukan!")
   
else:
    print(f"\n Ditemukan {len(report_files)} skenario model.")

for report_path in report_files:
    # Ambil nama model dari nama file
    filename = os.path.basename(report_path)
    model_name = filename.replace("_report.csv", "")
    folder_path = os.path.dirname(report_path)
    
    # Cari pasangan history-nya (untuk cek Epoch berhenti di mana)
    history_path = os.path.join(folder_path, f"{model_name}_history.csv")
    
    try:
        # --- A. BACA NILAI METRICS (Precision, Recall, dll) ---
        df_rep = pd.read_csv(report_path, index_col=0)
        
        # Logika pengambilan Akurasi yang aman
        acc = 0.0
        if 'accuracy' in df_rep.index:
            # Kadang accuracy ada di kolom f1-score atau support tergantung versi sklearn
            val = df_rep.loc['accuracy'].iloc[0] 
            if pd.isna(val) or val > 1.0: 
                 acc = df_rep.loc['accuracy', 'f1-score']
            else:
                 acc = val
        
        # Ambil Weighted Avg (Rata-rata tertimbang)
        precision = df_rep.loc['weighted avg', 'precision']
        recall = df_rep.loc['weighted avg', 'recall']
        f1_score = df_rep.loc['weighted avg', 'f1-score']
        
        # --- B. BACA JUMLAH EPOCH ---
        stopped_epoch = 0
        if os.path.exists(history_path):
            df_hist = pd.read_csv(history_path)
            stopped_epoch = len(df_hist) # Jumlah baris = Jumlah Epoch
        
        # --- C. MASUKKAN KE LIST ---
        recap_data.append({
            'Nama Skenario': model_name,
            'Akurasi (%)': round(acc * 100, 2),
            'Precision (%)': round(precision * 100, 2),
            'Recall (%)': round(recall * 100, 2),
            'F1-Score (%)': round(f1_score * 100, 2),
            'Stop Epoch': stopped_epoch
        })
        
    except Exception as e:
        print(f"Gagal membaca data untuk {model_name}: {e}")

# ==========================================
# 3. TAMPILKAN TABEL REKAP
# ==========================================
if recap_data:
    df_recap = pd.DataFrame(recap_data)

    # Urutkan berdasarkan Nama Skenario agar rapi (S1, S2, ...)
    df_recap.sort_values(by='Nama Skenario', inplace=True)

    print("\n" + "="*80)
    print("📊 TABEL PERBANDINGAN PERFORMA 12 MODEL")
    print("="*80)

    # Tampilkan Tabel dengan highlight warna
    # Semakin hijau = Semakin bagus
    styled_table = df_recap.style.background_gradient(cmap='Greens', subset=['Akurasi (%)', 'F1-Score (%)'])
    from IPython.display import display
    display(styled_table)

    # Simpan ke CSV untuk Skripsi
    df_recap.to_csv("Rekap_Data_Skripsi.csv", index=False)
    print("\n✅ File Excel tersimpan: 'Rekap_Data_Skripsi.csv' (Siap Download)")

    # ==========================================
    # 4. BUAT GRAFIK PERBANDINGAN (VISUAL)
    # ==========================================
    plt.figure(figsize=(12, 8))
    
    # Plot Bar Chart berdasarkan F1-Score
    sns.barplot(x='F1-Score (%)', y='Nama Skenario', data=df_recap, palette='viridis')
    
    # Tambahkan angka di ujung batang
    for index, value in enumerate(df_recap['F1-Score (%)']):
        plt.text(value + 0.5, index, f"{value}%", va='center', fontweight='bold', fontsize=10)

    plt.title('Perbandingan F1-Score: Mencari Model Terbaik', fontsize=16, fontweight='bold')
    plt.xlabel('F1-Score (%)')
    plt.ylabel('Skenario Model')
    plt.xlim(0, 110) # Beri ruang untuk teks
    plt.grid(axis='x', linestyle='--', alpha=0.5)
    plt.tight_layout()
    
    # Simpan Gambar
    plt.savefig("Grafik_Perbandingan_Model.png", dpi=300)
    print("✅ File Gambar tersimpan: 'Grafik_Perbandingan_Model.png' (Siap Download)")
    plt.show()

else:
    print("❌ Tidak ada data untuk ditampilkan.")